# Lesson 13 — 让电脑读取提醒器状态

这节课仍然延续桌面状态提醒器，但数据首次穿过**真实本机网络**。不需要 ESP32，也不需要 Python 基础：下面代码格是完整 C++ 程序。

## 我们要解决什么问题？

浏览器或电脑工具怎样询问桌面提醒器当前是否开启？

## 最小知识

TCP 提供的是有序字节流，不保留“消息”的天然边界。本课故意使用一行命令加换行作为最小帧格式。端口让同一台电脑上的不同服务可以区分彼此。

运行初始化格后，所有网络实验只绑定 `127.0.0.1`。Notebook 面板是真实客户端；业务逻辑、协议响应和状态都来自当前 C++ 代码格。

In [ ]:
# 运行支架：只执行一次。你写的网络逻辑仍是 C++，不需要学习 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))
from network_lab import register
register()


## 1. 第一个可运行实验

先点击“查询 STATUS”，再点击“发送 TOGGLE”，最后重新查询。注意三次操作是三条独立 TCP 连接：状态要保存于 C++ 服务进程，而不能放在一次 handler 的局部变量里。

运行下一格。面板的端口由 C++ 程序通过 `ACADEMY_READY` 报告；不要手填固定端口。

In [ ]:
%%network_lab l13-guided tcp guided pass
#include "academy/net/socket.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_tcp_line_server([&](const std::string& request) {
        if (request == "STATUS\n") {
            return std::string("reminder=") + (reminder_on ? "ON\n" : "OFF\n");
        }
        if (request == "TOGGLE\n") {
            reminder_on = !reminder_on;
            return std::string("reminder=") + (reminder_on ? "ON\n" : "OFF\n");
        }
        return std::string("ERR unknown command\n");
    });
}


## 2. 修改一次

把下一格中一个响应文本或状态初值改掉，再点击面板操作，观察改变从 C++ 穿过真实 localhost 连接回到输出。重新运行本格会终止旧进程并启动新进程。

In [ ]:
%%network_lab l13-modify tcp
#include "academy/net/socket.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_tcp_line_server([&](const std::string& request) {
        if (request == "STATUS\n") {
            return std::string("reminder=") + (reminder_on ? "ON\n" : "OFF\n");
        }
        if (request == "TOGGLE\n") {
            reminder_on = !reminder_on;
            return std::string("reminder=") + (reminder_on ? "ON\n" : "OFF\n");
        }
        return std::string("ERR unknown command\n");
    });
}


## 3. 故意弄坏

下面代码可编译，但**自动检查应失败**。先运行，点击“播放自动检查”，再根据红色结果解释问题。失败来自真实网络往返后的行为差异，不是 Notebook 预设动画。

In [ ]:
%%network_lab l13-broken tcp exercise fail
#include "academy/net/socket.hpp"

int main() {
    return academy::net::run_tcp_line_server([](const std::string& request) {
        bool reminder_on = false; // 每条连接都会重新创建它。
        if (request == "TOGGLE\n")
            reminder_on = !reminder_on;
        return std::string("reminder=") + (reminder_on ? "ON\n" : "OFF\n");
    });
}


## 4. 自己完成

在下方 C++ 中完成 TODO。先手动操作，再点“播放自动检查”。测试只编译这一格，不会读取 `solution/`。需要时再看 [分层 Hint](../lessons/13-tcp-status-service/hints.md)。

In [ ]:
%%network_lab l13-exercise tcp exercise
#include "academy/net/socket.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_tcp_line_server([&](const std::string& request) {
        // TODO: 支持 STATUS\n 和 TOGGLE\n；返回 reminder=ON\n 或 reminder=OFF\n。
        (void)request;
        return std::string("TODO\n");
    });
}


## 5. 你刚刚真正学到了什么？

你已经让提醒器通过真实 localhost 网络与另一个进程通信，并使用协议契约验证状态。ESP32 上可把同一条命令解析逻辑放在 Wi-Fi TCP server 的连接处理里。需要补上连接数、读超时、最大帧长和断线清理；host 结果不证明 Wi-Fi 射频可用。

完成尝试后才查看 [Solution Notebook](solutions/13-tcp-status-service-solution.ipynb)。